In [ ]:
!pip install -q transformers==4.36.2 accelerate==0.26.1 peft==0.13.2 bitsandbytes sentencepiece einops timm shortuuid
!git clone https://github.com/mbzuai-oryx/GeoChat.git
%cd GeoChat
!pip install -e . --no-deps -q

/content
Cloning into 'GeoChat'...
remote: Enumerating objects: 480, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 480 (delta 64), reused 38 (delta 38), pack-reused 400 (from 1)
Receiving objects: 100% (480/480), 63.84 MiB | 19.97 MiB/s, done.
Resolving deltas: 100% (155/155), done.
/content/GeoChat
Obtaining file:///content/GeoChat
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for geochat (pyproject.toml) ... done
  Created wheel for geochat: filename=geochat-1.1.1-0.editable-py3-none-any.whl size=8153 sha256=0a3250ec11012125400985742ea3530507289a62abd628fbbe20a7b87bfdcf77
  Stored in directory: /tmp/pip-ephem-wheel-cache-7lqhk27s/wheels/77/9d/36/859da510aaf3ae59c029ede61d673ddefe5c407ba86194b5a1
Successfully built geochat
     ━━━━━━━━━━

In [ ]:
from google.colab import files
uploaded = files.upload()  # select your checkpoint-50.zip

!mkdir -p /content/checkpoint-50
!unzip -q checkpoint-50.zip -d /content/checkpoint-50
!ls /content/checkpoint-50   # sanity check: should show adapter_config.json + adapter_model.safetensors

/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_no

Loading GeoChat......


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/749 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:600: UserWarning: for vision_model.embeddings.class_embedding: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(*args)
/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:600: UserWarning: for vision_model.embeddings.patch_embedding.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(*args)
/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:600: UserWarning: for vision_model.embeddings.posi

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
!pip install -q gdown
!gdown --folder https://drive.google.com/drive/folders/12eR0N3lZkdm1MSHUEf6j5Q1rtr1s-H-p?usp=sharing

In [ ]:
!sed -i 's/^from .language_model.geochat_mpt import.*/# &  # disabled - unused MPT path, breaks on current transformers/' /content/GeoChat/geochat/model/__init__.py
!cat /content/GeoChat/geochat/model/__init__.py   # confirm it's commented out

In [ ]:
import torch
from geochat.model.builder import load_pretrained_model
from geochat.mm_utils import get_model_name_from_path
from geochat.utils import disable_torch_init

disable_torch_init()

BASE_MODEL = "MBZUAI/geochat-7B"
model_name = get_model_name_from_path(BASE_MODEL)

tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path=BASE_MODEL,
    model_base=None,     # deliberately None - see note above, we apply
                          # the adapter ourselves right after this
    model_name=model_name,
    load_4bit=True,      # matches how you trained
    device="cuda",
)

from peft import PeftModel
model = PeftModel.from_pretrained(model, "/content/GeoChat/checkpoint-50")
model = model.eval()
print("Base model + your v3 adapter loaded.")

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:389: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Response: {<30><45><70><73>|<90>}
Detected Boxes: [{'label': None, 'box_2d': [180.0, 270.0, 420.0, 438.0], 'angle': 90}]


In [ ]:
import re
from PIL import Image, ImageDraw

from geochat.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN, DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN
from geochat.conversation import conv_templates, SeparatorStyle
from geochat.mm_utils import tokenizer_image_token, KeywordsStoppingCriteria

def parse_boxes(text):
    boxes = []
    for block in re.findall(r"\{<[^}]+>\}", text):
        nums = [int(x) for x in re.findall(r"-?\d+", block)]
        if len(nums) >= 4:
            boxes.append(nums[:4])
    return boxes

def test_grounding(image_path, query, mode="refer", max_new_tokens=256):
    image = Image.open(image_path).convert("RGB")

    if mode == "refer":
        qs = f"[refer] Give me the location of <p> {query} </p>"
    else:  # "detect_all"
        qs = f"[grounding]Detect all {query} in the image."

    if model.config.mm_use_im_start_end:
        qs = DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + DEFAULT_IM_END_TOKEN + '\n' + qs
    else:
        qs = DEFAULT_IMAGE_TOKEN + '\n' + qs

    conv = conv_templates["llava_v1"].copy()
    conv.append_message(conv.roles[0], qs)
    conv.append_message(conv.roles[1], None)
    prompt = conv.get_prompt()

    input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).cuda()
    image_tensor = image_processor.preprocess(
        [image], crop_size={'height': 504, 'width': 504}, size={'shortest_edge': 504}, return_tensors='pt'
    )['pixel_values']

    stop_str = conv.sep if conv.sep_style != SeparatorStyle.TWO else conv.sep2
    stopping_criteria = KeywordsStoppingCriteria([stop_str], tokenizer, input_ids)

    with torch.inference_mode():
        output_ids = model.generate(
            input_ids, images=image_tensor.half().cuda(),
            do_sample=False, num_beams=1, max_new_tokens=max_new_tokens,
            use_cache=True, stopping_criteria=[stopping_criteria],
        )

    raw = tokenizer.batch_decode(output_ids[:, input_ids.shape[1]:], skip_special_tokens=True)[0].strip()
    if raw.endswith(stop_str):
        raw = raw[: -len(stop_str)]
    print("Raw output:", raw)

    width, height = image.size
    annotated = image.copy()
    draw = ImageDraw.Draw(annotated)
    for x1, y1, x2, y2 in parse_boxes(raw):
        draw.rectangle([x1/100*width, y1/100*height, x2/100*width, y2/100*height], outline="red", width=3)

    return raw, annotated

{'text': '{<28><39><72><79>|<90>}', 'boxes': [{'label': None, 'box_2d': [168.00000000000003, 234.0, 432.0, 474.0], 'angle': 90}]}


In [ ]:
from google.colab import files
# uploaded = files.upload()  # upload a test image, e.g. plane.png

raw, annotated = test_grounding("/content/GeoChat/demo_images/04444.png", "basketball court", mode="detect_all")
annotated

Classification: Church


In [ ]:
import os
import shutil
import traceback

import torch
import nest_asyncio
import uvicorn
from fastapi import FastAPI, File, UploadFile, Form
from pyngrok import ngrok
from PIL import Image

from geochat.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN, DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN
from geochat.conversation import conv_templates, SeparatorStyle
from geochat.mm_utils import tokenizer_image_token, KeywordsStoppingCriteria

nest_asyncio.apply()
app = FastAPI()

ngrok.set_auth_token("3J96zcTo03XEJ2QnkYYwC2xGZdU_3DwXNCTPsYrN5ACQoWVxs")  # this token's been posted in plaintext a couple times now - regen it in ngrok's dashboard when you get a minute


import re
from collections import Counter

def parse_grounding_output(raw_text: str):
    """
    Parses GeoChat grounding output, defensively handling a known
    checkpoint-50 failure mode: repeating a full VQA-style sentence
    (e.g. "there are 13 houses") as the label on every box instead of a
    short category tag.

    Returns:
        text: a clean analysis string
        boxes: list of [x1, y1, x2, y2] on a 0-100 scale
    """
    entries = re.findall(r"<p>(.*?)</p>\s*(\{<[^}]+>\})", raw_text, flags=re.DOTALL)

    def extract_box(block):
        nums = [int(x) for x in re.findall(r"-?\d+", block)]
        return nums[:4] if len(nums) >= 4 else None

    if not entries:
        boxes = [b for b in (extract_box(blk) for blk in re.findall(r"\{<[^}]+>\}", raw_text)) if b]
        return raw_text.strip(), boxes

    labels = [label.strip() for label, _ in entries]
    boxes = [b for b in (extract_box(blk) for _, blk in entries) if b]

    most_common_label, count = Counter(labels).most_common(1)[0]
    looks_like_sentence = (
        len(most_common_label.split()) > 3
        or most_common_label.lower().startswith(("there are", "there is", "yes", "no"))
    )
    is_repeated = count >= max(2, int(len(labels) * 0.6))

    if looks_like_sentence and is_repeated:
        # VQA-style phrasing bled into the grounding template - surface it
        # once as the answer, boxes stand on their own
        return most_common_label, boxes

    # normal case: real distinct category labels
    unique_labels = list(dict.fromkeys(labels))
    return f"Found: {', '.join(unique_labels)}", boxes

def run_geochat(image_path: str, text_prompt: str, task_type: str = "vqa", max_new_tokens: int = 256):
    """Our own generation path - proper conv template + image token, which is
    exactly what was missing before and caused the garbage output."""
    image = Image.open(image_path).convert("RGB")

    if task_type == "grounding":
        qs = f"[grounding]{text_prompt}"
    elif task_type == "refer":
        qs = f"[refer] Give me the location of <p> {text_prompt} </p>"
    else:  # vqa / captioning
        qs = text_prompt

    if model.config.mm_use_im_start_end:
        qs = DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + DEFAULT_IM_END_TOKEN + '\n' + qs
    else:
        qs = DEFAULT_IMAGE_TOKEN + '\n' + qs

    conv = conv_templates["llava_v1"].copy()
    conv.append_message(conv.roles[0], qs)
    conv.append_message(conv.roles[1], None)
    prompt = conv.get_prompt()

    input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).cuda()
    image_tensor = image_processor.preprocess(
        [image], crop_size={'height': 504, 'width': 504}, size={'shortest_edge': 504}, return_tensors='pt'
    )['pixel_values']

    stop_str = conv.sep if conv.sep_style != SeparatorStyle.TWO else conv.sep2
    stopping_criteria = KeywordsStoppingCriteria([stop_str], tokenizer, input_ids)

    with torch.inference_mode():
        output_ids = model.generate(
            input_ids, images=image_tensor.half().cuda(),
            do_sample=False, num_beams=1, max_new_tokens=max_new_tokens,
            use_cache=True, stopping_criteria=[stopping_criteria],
        )

    text = tokenizer.batch_decode(output_ids[:, input_ids.shape[1]:], skip_special_tokens=True)[0].strip()
    if text.endswith(stop_str):
        text = text[: -len(stop_str)]
    return text


def _save_upload(file: UploadFile) -> str:
    os.makedirs("uploaded_images", exist_ok=True)
    image_path = os.path.join("uploaded_images", file.filename)
    with open(image_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)
    return image_path


@app.get("/")
def read_root():
    return {"message": "Hello from Google Colab"}


@app.post("/grounding")
async def grounding_endpoint(file: UploadFile = File(...), text_prompt: str = Form(...)):
    image_path = _save_upload(file)
    try:
        result = run_geochat(image_path, text_prompt, task_type="grounding")
    except Exception:
        traceback.print_exc()
        return {"text": "Error running grounding - check server logs"}
    print("DEBUG RESULT:", result, flush=True)
    return {"text": result}


@app.post("/vqa")
async def vqa_endpoint(file: UploadFile = File(...), text_prompt: str = Form(...)):
    image_path = _save_upload(file)
    try:
        result = run_geochat(image_path, text_prompt, task_type="vqa")
    except Exception:
        traceback.print_exc()
        return {"text": "Error running vqa - check server logs"}
    print("DEBUG RESULT:", result, flush=True)
    return {"text": result}


@app.post("/captioning")
async def caption_endpoint(file: UploadFile = File(...), text_prompt: str = Form(...)):
    image_path = _save_upload(file)
    try:
        result = run_geochat(image_path, text_prompt, task_type="captioning")
    except Exception:
        traceback.print_exc()
        return {"text": "Error running captioning - check server logs"}
    print("DEBUG RESULT:", result, flush=True)
    return {"text": result}


public_url = ngrok.connect(8000)
print(f"Public URL: {public_url}")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)
await server.serve()